# Recognizing Faces in the Wild - Kinship Verification

This notebook is designed for Kaggle Notebooks. It assumes the FIW competition dataset is mounted under `/kaggle/input/recognizing-faces-in-the-wild/` and writes generated artifacts under `/kaggle/working/`.

This repository also includes `main.ipynb`, a local/full workflow that downloads Kaggle data, trains and evaluates the model, and generates a Kaggle submission file. This Kaggle version focuses on the mounted-data scoring-head training and held-out evaluation workflow.

The pipeline embeds each face once with a frozen pretrained `facenet_pytorch.InceptionResnetV1(pretrained='vggface2')` encoder at its native 160×160 input size, trains a logistic-regression head on symmetric pair features, and scores kinship with a rank-blend of cosine similarity and the head logit. The inline pair-sampling and scoring code mirrors `pair_sampling.py` and `kinship.py` in the repository.

## Runtime assumptions

- The FIW competition dataset is attached as a Kaggle input at `/kaggle/input/recognizing-faces-in-the-wild/`.
- `/kaggle/working/` is writable and used for extracted images, saved model weights, and plots.
- Internet access or cached packages are available for `facenet_pytorch` and pretrained weights.
- GPU acceleration is recommended; runtime varies by Kaggle GPU type and package/download cache state.

## Data-use note

Before using the FIW competition data, review the Kaggle competition rules, especially the data-use requirements. Work using FIW data should cite the official FIW/RFIW papers listed in the competition documentation.


## Setup: Install dependencies

In [ ]:
!pip install facenet_pytorch -q
print('Dependencies installed.')

## Step 1: Load competition data from Kaggle input

In [ ]:
import os
import zipfile
import pandas
from collections import defaultdict
import glob

# Kaggle dataset path
dataset_path = '/kaggle/input/recognizing-faces-in-the-wild'

# Check structure
print('Dataset contents:')
for item in os.listdir(dataset_path):
    print(f'  {item}')

# Extract paths
train_zip = os.path.join(dataset_path, 'train-faces.zip')
test_zip = os.path.join(dataset_path, 'test-faces.zip')
relations_csv = os.path.join(dataset_path, 'train_relationships.csv')

# Use Kaggle's working directory for extraction
work_dir = '/kaggle/working'
train_faces_dir = os.path.join(work_dir, 'train-faces')
test_faces_dir = os.path.join(work_dir, 'test-faces')

# Extract training faces
if not os.path.exists(train_faces_dir):
    print('Extracting training faces...')
    with zipfile.ZipFile(train_zip, 'r') as z:
        z.extractall(work_dir)
    os.rename(os.path.join(work_dir, 'train_faces'), train_faces_dir)

# Extract test faces
if not os.path.exists(test_faces_dir):
    print('Extracting test faces...')
    with zipfile.ZipFile(test_zip, 'r') as z:
        z.extractall(work_dir)
    os.rename(os.path.join(work_dir, 'test_faces'), test_faces_dir)

print(f'Training faces: {len(os.listdir(train_faces_dir))} families')
print(f'Test faces: {len(os.listdir(test_faces_dir))} images')
print('Step 1 complete: Data loaded.')

## Step 2: Load and clean the labeled dataset

The `train_relationships.csv` file contains labeled kinship pairs in `family/member` format. Rows are removed when either member does not have corresponding image data in the extracted training faces directory.


In [ ]:
# Load relations
print(f'Loading {relations_csv}')
relations_df = pandas.read_csv(relations_csv, delimiter=',', header='infer')

# Create a dictionary to lookup image files for each member
family_dict = defaultdict(list)
for family in glob.glob(os.path.join(train_faces_dir, '*')):
    for member in glob.glob(os.path.join(family, '*')):
        for image_path in glob.glob(os.path.join(member, '*')):
            member_id = os.path.basename(member)
            image_file = os.path.basename(image_path)
            family_dict[member_id].append(image_file)

print(f'Images found for {len(family_dict)} members')

# Remove entries which do not exist in the training set
print(f'Original relations: {len(relations_df)} pairs')
fam_keys = family_dict.keys()
missing_relations_list = []

for index, row in relations_df.iterrows():
    split1 = row.p1.split('/')
    split2 = row.p2.split('/')
    p1fam, p1member = split1[0], split1[1]
    p2fam, p2member = split2[0], split2[1]
    
    if (p1fam not in fam_keys or p2fam not in fam_keys or 
        p1member not in family_dict[p1fam] or p2member not in family_dict[p2fam]):
        missing_relations_list.append(index)
        continue
    
    images1 = os.listdir(os.path.join(train_faces_dir, p1fam, p1member))
    images2 = os.listdir(os.path.join(train_faces_dir, p2fam, p2member))
    if len(images1) == 0 or len(images2) == 0:
        missing_relations_list.append(index)

if missing_relations_list:
    relations_df = relations_df.drop(missing_relations_list)
    print(f'Removed {len(missing_relations_list)} pairs with missing data')

print(f'Final relations: {len(relations_df)} pairs')
print('Step 2 complete: Data cleaned.')

## Step 3: Split by family and generate balanced pairs

Pairs are split by family so that members of the same family do not appear across training, validation, and held-out test splits. This reduces leakage from shared family identity across splits.

Positive pairs come from labeled kinship relationships. Negative pairs follow the repository's sampling policy (`pair_sampling.py`, issue #10): a candidate pair is never the same member, never a known relation in either direction, and never two members of the same family — family membership implies potential kinship. Positive and negative pair sets are asserted disjoint.

In [ ]:
import itertools
import random
import numpy
import torch

# --- Inline copies of the pair_sampling.py policy helpers -------------------

def build_excluded_pairs(relations_df, members):
    positive_member_pairs = set()
    for _, row in relations_df.iterrows():
        positive_member_pairs.add((row.p1, row.p2))
        positive_member_pairs.add((row.p2, row.p1))
    family_members = defaultdict(set)
    for m in members:
        family_members[m.split('/')[0]].add(m)
    same_family_pairs = set()
    for fam_members in family_members.values():
        for m1, m2 in itertools.permutations(fam_members, 2):
            same_family_pairs.add((m1, m2))
    return positive_member_pairs | same_family_pairs

def generate_negative_pairs(positives, candidate_members, member_images,
                            excluded_pairs, rng, max_attempts_multiplier=100):
    negatives = []
    attempts = 0
    max_attempts = len(positives) * max_attempts_multiplier
    while len(negatives) < len(positives) and attempts < max_attempts:
        attempts += 1
        p1 = rng.choice(candidate_members)
        p2 = rng.choice(candidate_members)
        if p1 == p2 or (p1, p2) in excluded_pairs:
            continue
        for img1, img2 in itertools.product(member_images[p1], member_images[p2]):
            negatives.append([img1, img2, 0.0])
            if len(negatives) >= len(positives):
                break
    return negatives

# --- Family-aware split and balanced pairs (mirrors kinship.build_pairs) ----

rng = random.Random(42)
numpy.random.seed(42)
torch.manual_seed(42)

all_families = sorted({p.split('/')[0] for p in relations_df['p1']})
rng.shuffle(all_families)
n = len(all_families)
split_families = {
    'train': set(all_families[:int(0.70 * n)]),
    'val': set(all_families[int(0.70 * n):int(0.85 * n)]),
    'test': set(all_families[int(0.85 * n):]),
}
print(f"Family split: {len(split_families['train'])} train, "
      f"{len(split_families['val'])} val, {len(split_families['test'])} test")

members = sorted({m for row in relations_df.values for m in row})
member_images = {}
for member in members:
    member_path = os.path.join(train_faces_dir, member)
    if os.path.isdir(member_path):
        files = os.listdir(member_path)
        if files:
            member_images[member] = [member + '/' + f for f in files]

excluded_pairs = build_excluded_pairs(relations_df, members)

splits = {}
for name, family_set in split_families.items():
    subset = relations_df[relations_df['p1'].apply(lambda p: p.split('/')[0] in family_set)]
    positives = []
    for _, row in subset.iterrows():
        for img1 in member_images.get(row.p1, []):
            for img2 in member_images.get(row.p2, []):
                positives.append([img1, img2, 1.0])
    candidates = [m for m in members
                  if m.split('/')[0] in family_set and m in member_images]
    negatives = generate_negative_pairs(positives, candidates, member_images,
                                        excluded_pairs, rng)
    assert not ({(p[0], p[1]) for p in positives} & {(n_[0], n_[1]) for n_ in negatives})
    data = positives + negatives[:len(positives)]
    rng.shuffle(data)
    splits[name] = data
    pos = sum(1 for p in data if p[2] == 1.0)
    print(f'{name.capitalize():5s}: {len(data)} pairs ({pos} pos + {len(data) - pos} neg)')

## Step 4: Embed each face once with the frozen encoder

Earlier versions fine-tuned the backbone end-to-end (test AUC 0.674); the unmodified frozen encoder outperforms that pipeline. The pretrained `InceptionResnetV1` (VGGFace2) stays frozen in eval mode at its native 160×160 input size — its `forward()` L2-normalizes embeddings — and each unique face image is embedded exactly once.

In [ ]:
import torch.nn.functional as F
import torchvision.transforms as transforms
from PIL import Image
from facenet_pytorch import InceptionResnetV1

eval_transform = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
encoder = InceptionResnetV1(pretrained='vggface2').eval().to(device)
for param in encoder.parameters():
    param.requires_grad = False

def embed_images(image_paths, image_root, batch_size=128):
    """Embed each image once; failed loads are flagged, never zero-filled."""
    embeddings, failed = {}, set()
    batch, names = [], []
    def flush():
        if not batch:
            return
        with torch.no_grad():
            out = encoder(torch.stack(batch).to(device))
        for name, emb in zip(names, out):
            embeddings[name] = emb.cpu()
        batch.clear(); names.clear()
    for i, path in enumerate(image_paths):
        try:
            batch.append(eval_transform(Image.open(os.path.join(image_root, path))))
            names.append(path)
        except Exception as error:
            print(f'[Warning] Failed to load {path}: {error}')
            failed.add(path)
            continue
        if len(batch) >= batch_size:
            flush()
        if i % 2048 == 0:
            print(f'  embedded {i}/{len(image_paths)}')
    flush()
    return embeddings, failed

unique_images = sorted({img for data in splits.values() for pair in data for img in pair[:2]})
print(f'Embedding {len(unique_images)} unique face images on {device}')
embeddings, failed = embed_images(unique_images, train_faces_dir)
print(f'{len(embeddings)} embeddings ready ({len(failed)} images failed to load)')

def stack_embeddings(pairs):
    """Drop pairs touching a failed image, then stack tensors."""
    kept = [p for p in pairs if p[0] in embeddings and p[1] in embeddings]
    if len(kept) < len(pairs):
        print(f'[Warning] Dropped {len(pairs) - len(kept)} pairs with missing embeddings')
    e1 = torch.stack([embeddings[p[0]] for p in kept])
    e2 = torch.stack([embeddings[p[1]] for p in kept])
    labels = numpy.array([p[2] for p in kept], dtype=numpy.float32)
    return e1, e2, labels

e1_train, e2_train, y_train = stack_embeddings(splits['train'])
e1_val, e2_val, y_val = stack_embeddings(splits['val'])
e1_test, e2_test, y_test = stack_embeddings(splits['test'])
print('Embeddings stacked for all splits.')

## Step 5: Train the logistic-regression scoring head

The only trained component: a single linear layer over the symmetric pair features `[|e1−e2|, e1⊙e2]`, optimized with binary cross-entropy and early-stopped on validation AUC.

In [ ]:
import torch.nn as nn
from sklearn.metrics import roc_auc_score

def pair_features(e1, e2):
    return torch.cat([(e1 - e2).abs(), e1 * e2], dim=1)

def logit_scores(head, e1, e2, batch_size=8192):
    head.eval()
    out = []
    with torch.no_grad():
        for start in range(0, len(e1), batch_size):
            features = pair_features(e1[start:start + batch_size],
                                     e2[start:start + batch_size])
            out.append(head(features).squeeze(1))
    return torch.cat(out).numpy()

torch.manual_seed(0)
head = nn.Linear(2 * e1_train.shape[1], 1)
optimizer = torch.optim.Adam(head.parameters(), lr=1e-2, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=200)
loss_fn = nn.BCEWithLogitsLoss()
targets = torch.from_numpy(y_train)

best_val, best_state, best_epoch = 0.0, None, -1
for epoch in range(200):
    head.train()
    perm = torch.randperm(len(e1_train))
    for start in range(0, len(perm), 1024):
        idx = perm[start:start + 1024]
        logits = head(pair_features(e1_train[idx], e2_train[idx]))
        loss = loss_fn(logits.squeeze(1), targets[idx])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    scheduler.step()
    val_auc = roc_auc_score(y_val, logit_scores(head, e1_val, e2_val))
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f'  epoch {epoch + 1:3d}  val AUC {val_auc:.4f}')
    if val_auc > best_val:
        best_val, best_epoch = val_auc, epoch
        best_state = {k: v.clone() for k, v in head.state_dict().items()}
    elif epoch - best_epoch >= 25:
        print(f'  early stop at epoch {epoch + 1} (best val AUC {best_val:.4f} @ epoch {best_epoch + 1})')
        break

head.load_state_dict(best_state)
torch.save(head.state_dict(), os.path.join(work_dir, 'logreg-head.pth'))
print(f'Scoring head trained (best val AUC {best_val:.4f}) and saved.')

## Step 6: Training-set sanity check

Related pairs should have higher cosine similarity between their frozen embeddings than unrelated pairs.

In [ ]:
cosine_train = F.cosine_similarity(e1_train, e2_train).numpy()
mean_related = cosine_train[y_train == 1.0].mean()
mean_unrelated = cosine_train[y_train == 0.0].mean()
print(f'Mean cosine similarity (related pairs):   {mean_related:.4f}')
print(f'Mean cosine similarity (unrelated pairs): {mean_unrelated:.4f}')
if mean_related > mean_unrelated:
    print('PASS: Related pairs are more similar than unrelated pairs.')
else:
    print('FAIL: Related pairs are NOT more similar.')

## Step 7: Evaluate on the held-out labeled test split

The final kinship score rank-blends cosine similarity with the head logit; the blend weight is selected on the validation split and applied unchanged to the test split.

AUC is threshold-independent. The threshold selected below is chosen on this same held-out split for inspection, so the derived accuracy, precision, and recall should be treated as exploratory diagnostics rather than separately validated deployment metrics.

In [ ]:
import numpy as np
from sklearn.metrics import roc_curve, accuracy_score, precision_score, recall_score

def normalized_ranks(scores):
    scores = np.asarray(scores)
    ranks = np.empty(len(scores), dtype=np.float64)
    ranks[np.argsort(scores, kind='stable')] = np.arange(1, len(scores) + 1)
    return ranks / len(scores)

def rank_blend(cos, logits, weight):
    return weight * normalized_ranks(cos) + (1 - weight) * normalized_ranks(logits)

cosine_val = F.cosine_similarity(e1_val, e2_val).numpy()
logit_val = logit_scores(head, e1_val, e2_val)
cosine_test = F.cosine_similarity(e1_test, e2_test).numpy()
logit_test = logit_scores(head, e1_test, e2_test)

blend_weight, blend_val_auc = 0.5, 0.0
for weight in np.arange(0.0, 1.01, 0.1):
    auc_w = roc_auc_score(y_val, rank_blend(cosine_val, logit_val, weight))
    if auc_w > blend_val_auc:
        blend_val_auc, blend_weight = auc_w, float(weight)
print(f'Blend weight (cosine share, selected on val): {blend_weight:.1f} (val AUC {blend_val_auc:.4f})')

scores = rank_blend(cosine_test, logit_test, blend_weight)
auc = roc_auc_score(y_test, scores)

fpr, tpr, thresholds = roc_curve(y_test, scores)
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]
predictions = (scores >= optimal_threshold).astype(float)

print(f'\n=== TEST RESULTS ===')
print(f'AUC — cosine only:      {roc_auc_score(y_test, cosine_test):.4f}')
print(f'AUC — logreg head only: {roc_auc_score(y_test, logit_test):.4f}')
print(f'AUC — rank blend:       {auc:.4f}')
print(f'Optimal Threshold:      {optimal_threshold:.4f}')
print(f'Accuracy:               {accuracy_score(y_test, predictions):.4f}')
print(f'Precision:              {precision_score(y_test, predictions):.4f}')
print(f'Recall:                 {recall_score(y_test, predictions):.4f}')

## Step 8: Visualize evaluation results

The ROC curve summarizes threshold-independent ranking quality on the held-out labeled test split. The similarity histogram shows whether related pairs tend to have higher embedding cosine similarity than unrelated pairs.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(fpr, tpr, label=f'ROC (AUC = {auc:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', label='Random')
axes[0].scatter(fpr[optimal_idx], tpr[optimal_idx], color='red', zorder=5,
                label=f'Optimal ({optimal_threshold:.2f})')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].hist(cosine_test[y_test == 1.0], bins=30, alpha=0.6,
             label=f'Related (n={int(y_test.sum())})')
axes[1].hist(cosine_test[y_test == 0.0], bins=30, alpha=0.6,
             label=f'Unrelated (n={int((1 - y_test).sum())})')
axes[1].set_xlabel('Cosine Similarity')
axes[1].set_ylabel('Count')
axes[1].set_title('Embedding Similarity Distributions')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(work_dir, 'evaluation.png'), dpi=100, bbox_inches='tight')
plt.show()

print('Evaluation plots saved.')

## Summary

- Embedded each face once with a frozen pretrained InceptionResnetV1/VGGFace2 encoder at 160×160.
- Generated balanced positive/negative kinship pairs from the cleaned relationship data under the repository's negative-sampling policy.
- Trained a logistic-regression scoring head on symmetric pair features, early-stopped on validation AUC.
- Evaluated a validation-weighted rank-blend of cosine similarity and head logits on a held-out family split.

**Outputs saved to `/kaggle/working/`:**

- `logreg-head.pth`: trained scoring-head weights.
- `evaluation.png`: ROC curve and similarity distributions.